### libraries

In [2]:
## libraries 

import pandas as pd
import os   
import numpy as np
import re
#from rapidfuzz import process, fuzz




### Functions and dictionaries

In [4]:
### dicitonary for the values 

### Missing codes

missing_code = {
    'N888': 'Not Applicable',
      'N999': 'Unknown',
        'N998': 'Subject Refused',
          'N997': 'Not Adminstered'
          }

fall_code = {
        1: "0",
        2: "1",
        3: "2",
        4: ">=3"
}


randomization = {
    '1.0': 'Augmentation with Aripiprazole',
      '2.0': 'Augmentation with Bupropion',
        '3.0': 'Switch to Bupropion',
          '4.0': 'Direct to Step 2'
          }


gender_code = {
    '1' : 'Male',
    '2' : 'Female'
}

race_code = {
    '1': 'Black/African American',
    '2': 'White',
    '3': 'Hawaiian/Pacific Islander',
    '4': 'Asian',
    '5': 'American Indian/Alaska Native (includes First Nations)',
    '6': 'Multi-race',
    '7': 'Other'
}

ethnicity_code = {
    '1': 'Hispanic/Latino',
    '2': 'Non-Hispanic'
}

frequency_code = {
    '1': 'Not taking it at all (missed all doses)',
    '2': 'Missed dose more often than not',
    '3': 'Missed dose more often than 2 times a week',
    '4': 'Missed dose more often than 1 time a week',
    '5': 'Missed dose 1 time a week',
    '6': 'Missed dose less than 1 time a week',
    '7': 'Never missed a dose'
}
medication_map = {
    # Bupropion family
    'BUPROPION': 'BUPROPION',
    'BUPROPRION': 'BUPROPION',
    'BUPROIPION': 'BUPROPION',
    'WELLBUTRIN': 'BUPROPION',
    'WELLBUTRIN 150MG': 'BUPROPION',
    'WELLBUTRIN 150MG QD': 'BUPROPION',
    'WELLBUTRIN 150MG ': 'BUPROPION',
    'WELLBUTRIN 150': 'BUPROPION',
    'WELLBUTRIN 450': 'BUPROPION',
    'WELLBUTRIN 450MG': 'BUPROPION',
    'WELLBUTRIN 300MG QD': 'BUPROPION',
    'WELLBUTRIN XL': 'BUPROPION',
    'BUPROPION EXTENDED RELEASE ORAL TABLET [WELLBUTRIN]': 'BUPROPION',

    # Escitalopram family
    'ESCITALOPRAM': 'ESCITALOPRAM',
    'LEXAPRO': 'ESCITALOPRAM',
    'LEXAPRO 10MG QD': 'ESCITALOPRAM',
    'LEXAPRO 20': 'ESCITALOPRAM',
    'ESCITAOLPRAM': 'ESCITALOPRAM',

    # Duloxetine family
    'DULOXETINE': 'DULOXETINE',
    'DULOXETING': 'DULOXETINE',
    'DULOXETIN': 'DULOXETINE',
    'CYMBALTA': 'DULOXETINE',
    'CYMBALTA 60MG QD': 'DULOXETINE',
    'CYMBALTA 90': 'DULOXETINE',

    # Venlafaxine family
    'VENLAFAXINE': 'VENLAFAXINE',
    'VENLAFAZINE': 'VENLAFAXINE',
    'EFFEXOR': 'VENLAFAXINE',
    'VENLAFAXINE 150MG QD': 'VENLAFAXINE',
    'EFFEXOR 150MG QD': 'VENLAFAXINE',
    'EFFEXOR 150MG': 'VENLAFAXINE',
    'EFFEXOR 150MG BID': 'VENLAFAXINE',

    # Fluoxetine family
    'FLUOXETINE': 'FLUOXETINE',
    'FLUOEXTINE': 'FLUOXETINE',
    'PROZAC': 'FLUOXETINE',
    'PROZAC 40MG': 'FLUOXETINE',
    'PROZAC 40': 'FLUOXETINE',

    # Citalopram family
    'CITALOPRAM': 'CITALOPRAM',
    'CELEXA': 'CITALOPRAM',
    'CITALOPRAM 20MG': 'CITALOPRAM',
    'CELEXA 20MG': 'CITALOPRAM',
    'CELEXA 40': 'CITALOPRAM',
    'CELEXA 10MG': 'CITALOPRAM',
    'CELEXA 10MG QD': 'CITALOPRAM',

    # Aripiprazole family
    'ARIPIPRAZOLE': 'ARIPIPRAZOLE',
    'ABILIFY': 'ARIPIPRAZOLE',
    'ABILIFY 2MG': 'ARIPIPRAZOLE',
    'ABILIFY 2': 'ARIPIPRAZOLE',
    'ABILFY': 'ARIPIPRAZOLE',
    'ABILIFY 2MG QD': 'ARIPIPRAZOLE',
    'ARIPIPRAZOLE/ABILIFY': 'ARIPIPRAZOLE',
    'ARIPIPRAZOLE 2 MG ORAL TABLET [ABILIFY]': 'ARIPIPRAZOLE',

    # Sertraline family
    'SERTRALINE': 'SERTRALINE',
    'SERTALINE': 'SERTRALINE',
    'ZOLOFT': 'SERTRALINE',
    'ZOLOFT 100MG QD': 'SERTRALINE',
    'ZOLOFT 50MG QD': 'SERTRALINE',
    'ZOLOFT 100MG': 'SERTRALINE',

    # Mirtazapine family
    'MIRTAZAPINE': 'MIRTAZAPINE',
    'REMERON 45MG': 'MIRTAZAPINE',
    'REMERON 15': 'MIRTAZAPINE',
    'MIRTAZAPINE 15 MG': 'MIRTAZAPINE',

    # Paroxetine family
    'PAROXETINE': 'PAROXETINE',
    'PAXIL': 'PAROXETINE',

    # Vortioxetine family (Brintellix / Trintellix)
    'VORTIOXETINE': 'VORTIOXETINE',
    'BRINTELLIX': 'VORTIOXETINE',
    'TRINTELLIX': 'VORTIOXETINE',

    # Trazodone
    'TRAZODONE': 'TRAZODONE',

    # Olanzapine
    'OLANZAPINE': 'OLANZAPINE',

    # Vilazodone
    'VILAZODONE': 'VILAZODONE',

    # Buspirone
    'BUSPIRONE': 'BUSPIRONE',

    # Nortriptyline
    'NORTRIPTYLINE': 'NORTRIPTYLINE',

    # Amitriptyline
    'AMITRIPTYLINE': 'AMITRIPTYLINE',

    # Desvenlafaxine (Pristiq)
    'DESVENLAFAXINE': 'DESVENLAFAXINE',
    'PRISTIQ': 'DESVENLAFAXINE',
    'DESVENLAFAXINE 50 MG [PRISTIQ]': 'DESVENLAFAXINE',

    # Quetiapine (Seroquel)
    'SEROQUEL': 'QUETIAPINE',
    'QUETIAPINE': 'QUETIAPINE',

    # Fluvoxamine
    'FLUVOXAMINE': 'FLUVOXAMINE',

    # Fetzima
    'FETZIMA': 'FETZIMA',

    # Klonopin
    'KLONOPIN': 'KLONOPIN',

    # Lamictal (Lamotrigine)
    'LAMICTAL 100MG': 'LAMOTRIGINE',

    # Vybrid
    'VYBRID 40MG': 'VYBRID',
    
     # PROZAC additional dosages
    'PROZAC 20MG EVERY OTHER DAY': 'FLUOXETINE',
    'PROZAC 30MG': 'FLUOXETINE',
    'PROZAC 40MG QD': 'FLUOXETINE',

    # Lexapro variants
    'LEXAPRO 10MG': 'ESCITALOPRAM',
    'LEXAPRO 20MG QD': 'ESCITALOPRAM',

    # Bupropion variants
    'WELLBUTRIN 150 QD': 'BUPROPION',
    'WELLBUTRIN XR 150': 'BUPROPION',
    'WELLBURIN 150MG QD': 'BUPROPION',
    'WELLBUTRIN 300MG': 'BUPROPION',

    # Venlafaxine variants
    'VENLAFAXINE XR 150MG QD': 'VENLAFAXINE',
    'VENLAFAXINE 150MG BID': 'VENLAFAXINE',

    # Duloxetine misspelling
    'CYMBALA 60MG QD': 'DULOXETINE',

    # Celexa variant
    'CELEXA 20MG QD': 'CITALOPRAM',

    # Aripiprazole variants
    'ABILIFY 5MG QD': 'ARIPIPRAZOLE',
    'ARIPIRAZOLE': 'ARIPIPRAZOLE',
    'ARIPRIPRAZOLE': 'ARIPIPRAZOLE',
    'ARIPIPRAZOLE 5 MG ORAL TABLET [ABILIFY]': 'ARIPIPRAZOLE',

    # Escitalopram misspelling
    'ESSCITALOPRAM': 'ESCITALOPRAM',

    # Nortriptyline
    'NORTRIPTYLINE': 'NORTRIPTYLINE',

    # Fluvoxamine
    'FLUVOXAMINE': 'FLUVOXAMINE',

    # Desipramine
    'DESIPRAMINE': 'DESIPRAMINE',

    # Lorazepam
    'LORAZEPAM': 'LORAZEPAM',

    # Vortioxetine
    'VORTIOXETINE': 'VORTIOXETINE',

    
    'BUPRIOPION': 'BUPROPION',
    'BUBROPION': 'BUPROPION',
    'BUPROPION 300MG QD': 'BUPROPION',

    # Escitalopram misspelling
    'ESCITALPRAM': 'ESCITALOPRAM',

    # Aripiprazole misspellings
    'ARIPRAZAOLE': 'ARIPIPRAZOLE',
    'ARIPIPRAOZOLE': 'ARIPIPRAZOLE',

    # Fluoxetine formatted with generic/brand
    'FLUOXETINE (PROZAC)': 'FLUOXETINE',

    # Citalopram dosage variant
    'CITALOPRAM 20MG QD': 'CITALOPRAM',

    # Venlafaxine dosage variants
    'VENLAFAXINE 300MG': 'VENLAFAXINE',
    'VENLAFAXINE 150MG': 'VENLAFAXINE',

    # Bupropion spelling and dosing variants
    'BUPROPION 150MG': 'BUPROPION',
    'BUPROPION 150': 'BUPROPION',
    'WELLBUTRIN 300MG DAILY': 'BUPROPION',
    'WELLBUTRIN 300': 'BUPROPION',
    'WELLBUTRIN 450MG QD': 'BUPROPION',
    'BUPROPRION ': 'BUPROPION',
    'WELLBUTRING': 'BUPROPION',

    # Escitalopram misspelling
    'EXCITALOPRAM': 'ESCITALOPRAM',

    # Aripiprazole variants
    'ABILIFY 5MG': 'ARIPIPRAZOLE',
    'ABILIFY 8MG QD': 'ARIPIPRAZOLE',
    'ARIPRAZOLE': 'ARIPIPRAZOLE',
    'ARIPIPRAZOLE ': 'ARIPIPRAZOLE',

    # Duloxetine misspellings and dosing
    'CYMBALTA 60MG': 'DULOXETINE',
    'CYMBATA 60MM': 'DULOXETINE',

    # Mirtazapine dosing
    'REMERON 45': 'MIRTAZAPINE',

    # Vilazodone dosing
    'VILAZODONE 40MG': 'VILAZODONE',

    # Venlafaxine dosing
    'EFFEXOR 150': 'VENLAFAXINE',
    # DULOXETINE variants
    'DULOEXTINE': 'DULOXETINE',

    # BUPROPION variants
    'BUPROPION 450MG QD': 'BUPROPION',
    'BUPROPION ': 'BUPROPION',

    # ESCITALOPRAM variants
    'ESCITALOPRAM LEXAPRO 10MG': 'ESCITALOPRAM',

    # ARIPIPRAZOLE variants
    'ABILIFY 5 MG': 'ARIPIPRAZOLE',
    'ABILIFY 10MG': 'ARIPIPRAZOLE',
    'ABILIFY 2MG DAILY': 'ARIPIPRAZOLE',
    'ARIPIPRAZOLE 7.5 MG/ML [ABILIFY]': 'ARIPIPRAZOLE',
    'ARIPIPROZOLE': 'ARIPIPRAZOLE',

    # VENLAFAXINE variants
    'VENLAFAZXINE': 'VENLAFAXINE',
    'VENLAFAXINE 300MG QD': 'VENLAFAXINE',

    # FLUOXETINE variants
    'FLOXETINE': 'FLUOXETINE',
    'PROZAC 20': 'FLUOXETINE',

    # SERTRALINE misspellings
    'SETRALINE': 'SERTRALINE',

    # VILAZODONE variants
    'VIIBRYD': 'VILAZODONE',

    # CYMBALTA dosing
    'CYMBALTA 90 MG': 'DULOXETINE',

    # CELEXA dosing
    'CELEXA 40MG': 'CITALOPRAM',

    # Other drugs not previously captured
    'RISPERIDONE': 'RISPERIDONE',
    'CLONAZEPAM': 'CLONAZEPAM',
    'GABAPENTIN': 'GABAPENTIN',
    'FLUVOXAMINE': 'FLUVOXAMINE',

    'ARIPIPRAZOLE 2MG': 'ARIPIPRAZOLE',
    'ARIPIPRAZOLE 2MG QD': 'ARIPIPRAZOLE',

    # VENLAFAXINE with trailing space
    'VENLAFAXINE ': 'VENLAFAXINE',

    # CELEXA numeric short form
    'CELEXA 20': 'CITALOPRAM',

     'ARIPIPRAZOLA': 'ARIPIPRAZOLE',

    # PAROXETINE with trailing space
    'PAROXETINE ': 'PAROXETINE',

    # ABILIFY with extra space and number only
    'ABILIFY  5': 'ARIPIPRAZOLE',

    # LAMICTAL alternative naming
    'LAMICTAL': 'LAMOTRIGINE',

    # CITALOPRAM short dosage
    'CITALOPRAM 20': 'CITALOPRAM',

    # REMERON brand to MIRTAZAPINE
    'REMERON': 'MIRTAZAPINE',

    # BUPROPION SR formulation
    'WELLBUTRIN SR': 'BUPROPION',

    # ZOLOFT dosage variant
    'ZOLOFT 150MG QD': 'SERTRALINE',

    # VENLAFAXINE dosage variant
    'VENLAFAXINE 75MG QD': 'VENLAFAXINE',

    # DULOXETINE with trailing space
    'DULOXETINE ': 'DULOXETINE',

    # CIPRALEX brand to ESCITALOPRAM
    'CIPRALEX': 'ESCITALOPRAM',

    # VENLAFAXINE misspelling
    'VENLAFZXINE': 'VENLAFAXINE',

    # VILAZODONE (brand and generic already mapped consistently as VILAZODONE)
    'VILAZODONE': 'VILAZODONE',

    'BURPROPION': 'BUPROPION',
    'ESCITALOPRAM ': 'ESCITALOPRAM',
    'ABILFIY 5MG QD': 'ARIPIPRAZOLE',

    # Aliases and branded names
    'BUPROPION (WELLBUTRIN)': 'BUPROPION',
    'BUPROPION SR': 'BUPROPION',
    'CYMBALTA 90MG QD': 'DULOXETINE',
    'FETZIMA': 'LEVOMILNACIPRAN',  
    'MEMANTINE': 'MEMANTINE',
    'DOXEPIN': 'DOXEPIN',
    # lithium
    'Lithium ': 'Lithium',
    # Doses recorded instead of names
    '150': 'UNKNOWN',
    '300': 'UNKNOWN',

    # ARIPIPRAZOLE variants
    'ARIPIPRAZOLE 5MG QD': 'ARIPIPRAZOLE',
    'ABILIFY 7.5MG QD': 'ARIPIPRAZOLE',
    
    'BURPROPION': 'BUPROPION',
    'BUPROPION (WELLBUTRIN)': 'BUPROPION',
    'BUPROPION 150': 'BUPROPION',
    'BUPROPION SR': 'BUPROPION',

    # ARIPIPRAZOLE variants
    'ARIPIPRAZOLE 5MG QD': 'ARIPIPRAZOLE',
    'ABILFIY 5MG QD': 'ARIPIPRAZOLE',
    'ABILIFY 7.5MG QD': 'ARIPIPRAZOLE',

    # ESCITALOPRAM with trailing space
    'ESCITALOPRAM ': 'ESCITALOPRAM',

    # CYMBALTA variant
    'CYMBALTA 90MG QD': 'DULOXETINE',

    # Known valid medications not yet explicitly mapped
    'MEMANTINE': 'MEMANTINE',
    'DOXEPIN': 'DOXEPIN',

    # FETZIMA already mapped previously to itself, include for completeness
    'FETZIMA': 'FETZIMA',

    # Venlafaxine variant
    'VENLAFAXINE 225MG': 'VENLAFAXINE',
    'VENLAFIXINE': 'VENLAFAXINE',

    # Paroxetine variant (Paxil is paroxetine)
    'PAXIL 60MG': 'PAROXETINE',

    # Aripiprazole variants
    'ABILIFY 2MG QOD': 'ARIPIPRAZOLE',
    'ABILFY 4MG QD': 'ARIPIPRAZOLE',
    'ARPIPRAZOLE': 'ARIPIPRAZOLE',

    # Zopiclone (sleep aid)
    'ZOPICLONE': 'ZOPICLONE',

     'DULOXETNE': 'DULOXETINE',

    # BUPROPION variants
    'WELLBUTRIN ': 'BUPROPION',
    'BUPROPION (WELLBUTRIN) 300MG QD': 'BUPROPION',

    # VENLAFAXINE XR variant
    'VENLAFAXINE XR 225': 'VENLAFAXINE',

    # ABILIFY dosing variants
    'ABILIFY 7.5MG': 'ARIPIPRAZOLE',
    'ABILIFY 10MG QD': 'ARIPIPRAZOLE',
    'ABILIFY 4MG QD': 'ARIPIPRAZOLE',

    'DESIPRAMINE': 'DESIPRAMINE',
    'BUPROPION PILL': 'BUPROPION',
    'BUPROPION HYDROCHLORIDE 150 MG [WELLBUTRIN]': 'BUPROPION',
    'BUPROPION HYDROCHLORIDE 150 MG [BUPROBAN]': 'BUPROPION',
    'BUPROPION HYDROCHLORIDE 150 MG [BUDEPRION]': 'BUPROPION',
    'BUPROPION HYDROCHLORIDE 150 MG': 'BUPROPION',
    'BUPROPION ORAL TABLET': 'BUPROPION',

    'ESCITALOPRAM 5 MG': 'ESCITALOPRAM',
    'ESCITALOPRAM 10 MG': 'ESCITALOPRAM',
    'ESCITALOPRAM 20 MG ORAL TABLET [LEXAPRO]': 'ESCITALOPRAM',
    'ESCITALOPRAM PILL': 'ESCITALOPRAM',

   
    'VENLAFAXINE 150 MG [EFFEXOR]': 'VENLAFAXINE',
    'VENLAFAXINE ORAL TABLET': 'VENLAFAXINE',
    'VENLAFAXINE PILL': 'VENLAFAXINE',
    'EFFEXOR PILL': 'VENLAFAXINE',

    'FLUOXETINE 20 MG ORAL CAPSULE [PROZAC]': 'FLUOXETINE',

    'CITALOPRAM 40 MG': 'CITALOPRAM',

    'ARIPIPRAZOLE 2 MG ORAL TABLET': 'ARIPIPRAZOLE',
    'ARIPIPRAZOLE ORAL TABLET': 'ARIPIPRAZOLE',

    'MIRTAZAPINE PILL': 'MIRTAZAPINE',

    'VILAZODONE HYDROCHLORIDE 40 MG': 'VILAZODONE',

    'VORTIOXETINE': 'VORTIOXETINE',
     'BUPROPION': 'BUPROPION',

    'ESCITALOPRAM 10 MG [LEXAPRO]': 'ESCITALOPRAM',

    'DULOXETINE 30 MG': 'DULOXETINE',
    'DULOXETINE 60 MG': 'DULOXETINE',
    'DULOXETINE PILL': 'DULOXETINE',
    'DULOXETINE ORAL PRODUCT': 'DULOXETINE',

    'VENLAFAXINE 100 MG ORAL TABLET': 'VENLAFAXINE',

    'FLUOXETINE 60 MG ORAL CAPSULE': 'FLUOXETINE',

    'CITALOPRAM 10 MG ORAL CAPSULE': 'CITALOPRAM',
    'CITALOPRAM 20 MG ORAL TABLET [CELEXA]': 'CITALOPRAM',

    'ARIPIPRAZOLE 2 MG': 'ARIPIPRAZOLE',

    'SERTRALINE ORAL TABLET [ZOLOFT]': 'SERTRALINE',
    'SERTRALINE 150 MG': 'SERTRALINE',
    'SERTRALINE PILL': 'SERTRALINE',
    'SERTRALINE ORAL TABLET': 'SERTRALINE',

    'MIRTAZAPINE 45 MG ORAL TABLET': 'MIRTAZAPINE',
    'MIRTAZAPINE ORAL TABLET [REMERON]': 'MIRTAZAPINE',

    'DESVENLAFAXINE SUCCINATE 100 MG [PRISTIQ]': 'DESVENLAFAXINE',
    'DESVENLAFAXINE ORAL PRODUCT': 'DESVENLAFAXINE',

    'QUETIAPINE': 'QUETIAPINE',

    'FLUVOXAMINE ORAL TABLET': 'FLUVOXAMINE',

    'FETZIMA': 'LEVOMILNACIPRAN',

    'BUSPIRONE': 'BUSPIRONE',

    'VORTIOXETINE': 'VORTIOXETINE',

    'AMITRIPTYLINE': 'AMITRIPTYLINE',

    'NORTRIPTYLINE': 'NORTRIPTYLINE',

    'TRAZODONE': 'TRAZODONE',

    'KLONOPIN': 'CLONAZEPAM',

    'LAMOTRIGINE': 'LAMOTRIGINE',

    'WELLBUTRIN PILL': 'BUPROPION',

    'ESCITALOPRAM 20 MG [LEXAPRO]': 'ESCITALOPRAM',

    'FLUOXETINE 10 MG': 'FLUOXETINE',

    'VENLAFAXINE EXTENDED RELEASE ORAL CAPSULE [EFFEXOR]': 'VENLAFAXINE',

    'SERTRALINE 150 MG ORAL TABLET': 'SERTRALINE',

    'MIRTAZAPINE 30 MG [REMERON]': 'MIRTAZAPINE',

    'VORTIOXETINE': 'VORTIOXETINE',

    'DESIPRAMINE': 'DESIPRAMINE',

    'LORAZEPAM': 'LORAZEPAM',

    'NORTRIPTYLINE': 'NORTRIPTYLINE',

    'FLUVOXAMINE': 'FLUVOXAMINE',

    'PAROXETINE HYDROCHLORIDE 20 MG [PAXIL]': 'PAROXETINE',

    'ESCITALOPRAM 5 MG ORAL TABLET': 'ESCITALOPRAM',

    'VENLAFAXINE 150 MG': 'VENLAFAXINE',

    'QUETIAPINE': 'QUETIAPINE',

    'VORTIOXETINE': 'VORTIOXETINE',

    'TRAZODONE': 'TRAZODONE',

    'VILAZODONE': 'VILAZODONE',

    'NORTRIPTYLINE': 'NORTRIPTYLINE',

    'AMITRIPTYLINE': 'AMITRIPTYLINE',

    'LAMOTRIGINE': 'LAMOTRIGINE',

    'BUPROPION HYDROCHLORIDE 300 MG': 'BUPROPION',

    'VENLAFAXINE 225 MG': 'VENLAFAXINE',

    'VILAZODONE HYDROCHLORIDE 40 MG ORAL TABLET [VIIBRYD]': 'VILAZODONE',

    'CITALOPRAM ORAL TABLET': 'CITALOPRAM',

    'ARIPIPRAZOLE 5 MG [ABILIFY]': 'ARIPIPRAZOLE',

    'VORTIOXETINE': 'VORTIOXETINE',

    'DESIPRAMINE': 'DESIPRAMINE',

    'NORTRIPTYLINE': 'NORTRIPTYLINE',

    'BUPROPION HYDROCHLORIDE 300 MG [WELLBUTRIN]': 'BUPROPION',

    'ARIPIPRAZOLE 5 MG ORAL TABLET WITH SENSOR [ABILIFY]': 'ARIPIPRAZOLE',

    'CITALOPRAM PILL': 'CITALOPRAM',

    'LEVOMILNACIPRAN': 'LEVOMILNACIPRAN',

    'MEMANTINE': 'MEMANTINE',

    'DOXEPIN': 'DOXEPIN',

    'CLONAZEPAM': 'CLONAZEPAM',

    'ARIPIPRAZOLE 10 MG': 'ARIPIPRAZOLE',

    'VORTIOXETINE': 'VORTIOXETINE',

    'ARIPIPRAZOLE 10 MG ORAL TABLET [ABILIFY]': 'ARIPIPRAZOLE',

    'VORTIOXETINE': 'VORTIOXETINE',

    'ZOPICLONE': 'ZOPICLONE',

    'ARIPIPRAZOLE 5 MG ORAL TABLET': 'ARIPIPRAZOLE',
    'ARIPIPRAZOLE PILL': 'ARIPIPRAZOLE',

    'BUPROPION ORAL PRODUCT': 'BUPROPION',

    'VORTIOXETINE': 'VORTIOXETINE',

    'RISPERIDONE': 'RISPERIDONE',

    'CLONAZEPAM': 'CLONAZEPAM',

    'GABAPENTIN': 'GABAPENTIN',

    'FLUVOXAMINE': 'FLUVOXAMINE',

    'MIRTAZAPINE 15 MG DISINTEGRATING ORAL TABLET [REMERON]': 'MIRTAZAPINE',

    'VILAZODONE HYDROCHLORIDE 40 MG ORAL TABLET': 'VILAZODONE',

    'ABILIFY PILL': 'ARIPIPRAZOLE',

    'CITALOPRAM 40 MG ORAL TABLET [CELEXA]': 'CITALOPRAM',

    'FLUOXETINE 40 MG': 'FLUOXETINE',

    'LEVOMILNACIPRAN': 'LEVOMILNACIPRAN',

    'DESIPRAMINE': 'DESIPRAMINE',
   
}



def remove_missing_codes(df, column):
    """
    Remove rows where the specified column contains any of the defined missing codes.
    """
    if column not in df.columns:
        raise ValueError(f"Column '{column}' does not exist in the DataFrame.")
    
    return df[~df[column].astype(str).str.strip().isin(missing_code.keys())]

def remove_empty(df, column):
    """
    Remove rows where the specified column is NaN or an empty string.
    """
    return df[df[column].notna() & (df[column].astype(str).str.strip() != '')]

def clean_up_df(df, columns=None, remove_duplicates=True):
    """
    Clean up the DataFrame by:
    1. Replacing missing codes with their dictionary values across all columns
    2. Removing empty values only from specified column(s)
    3. Optionally handling duplicate record_ids
    
    Parameters:
    -----------
    df : pandas.DataFrame
        The input DataFrame
    columns : str or list of str, optional
        Column(s) to remove empty values from. If None, no empty value removal
    remove_duplicates : bool, default=True
        Whether to remove duplicate record_ids
        
    Returns:
    --------
    pandas.DataFrame
        Cleaned DataFrame
    """
    # Make a copy to avoid modifying the original DataFrame
    df_clean = df.copy()
    
    # Replace blank strings with NaN
    df_clean = df_clean.applymap(lambda x: pd.NA if isinstance(x, str) and x.strip() == "" else x)
    
    # First, replace all missing codes across the entire DataFrame
    initial_missing = df_clean.isin(missing_code.keys()).sum().sum()
    df_clean = df_clean.replace(missing_code)
    print(f"Total missing codes replaced across DataFrame: {initial_missing}")
    
    # Then handle empty values only for specified columns
    if columns:
        if isinstance(columns, str):
            columns = [columns]
            
        for column in columns:
            if column not in df_clean.columns:
                print(f"Warning: Column '{column}' not found in DataFrame")
                continue
                
            initial_rows = df_clean.shape[0]
            df_clean = remove_empty(df_clean, column)
            removed = initial_rows - df_clean.shape[0]
            print(f"Column '{column}': {removed} empty values removed")
    
    # Handle duplicates in record_id if requested and if record_id exists
    if remove_duplicates and 'record_id' in df_clean.columns and df_clean['record_id'].duplicated().any():
        print(f"Found {df_clean['record_id'].duplicated().sum()} duplicate record_ids")
        
        # Check if there's a date column
        date_cols = [col for col in df_clean.columns if 'date' in col.lower()]
        
        if date_cols:
            print(f"Using {date_cols[0]} to keep earliest entry for duplicates")
            df_clean = df_clean.sort_values(date_cols[0]).groupby('record_id').first().reset_index()
        else:
            print("Keeping most complete entry for duplicates")
            df_clean = df_clean.loc[df_clean.groupby('record_id').apply(
                lambda x: x.isnull().sum(axis=1).idxmin()
            )].reset_index(drop=True)
        
        print(f"DataFrame shape after removing duplicates: {df_clean.shape}")
    
    return df_clean

# fix dates

def standardize_dates(date_series):
    """
    Convert mixed MM-DD-YYYY or MM/DD/YYYY formats to standardized YYYY-MM-DD strings.
    Leaves already-ISO-formatted dates untouched.
    """
    # Step 1: replace slashes with dashes
    cleaned = date_series.astype(str).str.strip().str.replace('/', '-', regex=False)

    # Step 2: try parsing MM-DD-YYYY first, and fallback to coercion
    parsed = pd.to_datetime(cleaned, format='%m-%d-%Y', errors='coerce')

    # Step 3: If parsing failed, try general parsing (e.g., for already-correct formats)
    parsed_fallback = pd.to_datetime(cleaned, errors='coerce')

    # Step 4: combine the two attempts
    parsed_final = parsed.combine_first(parsed_fallback)

    return parsed_final

# fix numeric columns
def clean_numeric_column(series):
    """
    Converts numeric-looking values like 1.0 or '1.0' to integer strings ('1'),
    while preserving non-numeric strings like 'Unknown'.
    """
    return (
        series.astype(str)                            # Convert to string
              .str.strip()                           # Remove leading/trailing spaces
              .str.replace(r'\.0$', '', regex=True)   # Remove trailing .0 if present
              .where(~series.astype(str).str.lower().isin(['nan', 'none', '']), None)  # Standardize missing
    )



### NeuroPsych data processing

In [5]:
## Load up  neuropsych data

neuropsych_data = pd.read_excel("/projects/aabdulrasul/BAARD/BAARD/neurocog/ON_DATASET_11.8.24.xlsx",engine="openpyxl")
### pull out index score columns ("AIS_01",      "MDMIS_01",     "LIS_01",     "MVCIS_01",    "IMIS_01", 
#  "MTOTALIS_01", "INDEXSUM_01") and ID_complete and rename to record_id
neuropsych_data = neuropsych_data[["ID_complete", "Ref_date", "AIS_01", "MDMIS_01", "LIS_01", "MVCIS_01",
 "IMIS_01",  "MTOTALIS_01",'CWI3CSSFinal_01','DERRSS4_01','CWI4CSSFinal_01','DTMT4ER_01','DTMT4CO_01','DTMTS4_01',
 'RCS_Z_01',	'RDS_Z_01',	'RFC_Z_01',	'RFR_Z_01',	'RLO_Z_01',	'RLL_Z_01',	'RREC_Z_01'	,'RSR_Z_01','RSF_Z_01','RSM_Z_01', 


'DITTTC_01',
'DISTTTC_01',
'DTMT4_01',
'DTMT5_01',

'PICTURE_01',
'SEMFLU_01',
'DSPAN_01',
'CODING_01',
'LRECALL_01',
'LRECOG_01',
'SRECALL_01',
'MFIRECAL_01',
'MVE1_01',
'MVE2_01',
'MVECC_01',
'MVECN_01',
'MVECH_01',
'MNAM1_01',
'MNAM2_01',
'MNAM3_01',
'MMEM1_01',
'MMEM2_01',
'MADS_01',
'MALET_01',
'MASER7_01',
'MLREP_01',
'MLFLUEN_01',
'MABST_01',
'MDRNC_01',
'MDRCC_01',
'MDRMCC_01',
'MODATE_01',
'MOMON_01',
'MOYEAR_01',
'MODAY_01',
'MOPL_01',
'MOCITY_01',
'MO1ED_01',
'MFLTOT_01',
'MFLP_01',
'MFLI_01'

]]
neuropsych_data = neuropsych_data.rename(columns={"ID_complete": "record_id"})

# make NA for where MFLTOT_01, LINE_01, PICTURE_01, SEMFLU_01, DSPAN_01, CODING_01, LRECALL_01, LRECOG_01, SRECALL_01 are 95
neuropsych_data.loc[neuropsych_data['DITTTC_01'] > 900,'DITTTC_01'] = pd.NA
neuropsych_data.loc[neuropsych_data['DISTTTC_01'] > 900,'DISTTTC_01'] = pd.NA
neuropsych_data.loc[neuropsych_data['DTMT4_01'] > 900,'DTMT4_01'] = pd.NA
neuropsych_data.loc[neuropsych_data['DTMT5_01'] > 900,'DTMT5_01'] = pd.NA
neuropsych_data.loc[neuropsych_data['CWI4CSSFinal_01'] > 900,'CWI4CSSFinal_01'] = pd.NA

neuropsych_data.loc[neuropsych_data['PICTURE_01'] > 90,'PICTURE_01'] = pd.NA
neuropsych_data.loc[neuropsych_data['SEMFLU_01'] > 90, 'SEMFLU_01'] = pd.NA
neuropsych_data.loc[neuropsych_data['DSPAN_01'] > 90, 'DSPAN_01'] = pd.NA
neuropsych_data.loc[neuropsych_data['CODING_01'] > 90, 'CODING_01'] = pd.NA
neuropsych_data.loc[neuropsych_data['LRECALL_01'] > 90, 'LRECALL_01'] = pd.NA
neuropsych_data.loc[neuropsych_data['LRECOG_01'] > 90, 'LRECOG_01'] = pd.NA
neuropsych_data.loc[neuropsych_data['SRECALL_01'] > 90, 'SRECALL_01'] = pd.NA
neuropsych_data.loc[neuropsych_data['MFIRECAL_01'] > 90, 'MFIRECAL_01'] = pd.NA
neuropsych_data.loc[neuropsych_data['DTMTS4_01'] > 90, 'DTMTS4_01'] = pd.NA
neuropsych_data.loc[neuropsych_data['CWI3CSSFinal_01'] > 90,'CWI3CSSFinal_01'] = pd.NA

# export to csv
neuropsych_data.to_csv("/projects/aabdulrasul/BAARD/BAARD/temp/processed/baseline_indexscores.csv", index=False)

In [10]:
neuropsych_data.columns.tolist()

['record_id',
 'Ref_date',
 'AIS_01',
 'MDMIS_01',
 'LIS_01',
 'MVCIS_01',
 'IMIS_01',
 'MTOTALIS_01',
 'CWI3CSSFinal_01',
 'DERRSS4_01',
 'CWI4CSSFinal_01',
 'DTMT4ER_01',
 'DTMT4CO_01',
 'DTMTS4_01',
 'RCS_Z_01',
 'RDS_Z_01',
 'RFC_Z_01',
 'RFR_Z_01',
 'RLO_Z_01',
 'RLL_Z_01',
 'RREC_Z_01',
 'RSR_Z_01',
 'RSF_Z_01',
 'RSM_Z_01',
 'DITTTC_01',
 'DISTTTC_01',
 'DTMT4_01',
 'DTMT5_01',
 'PICTURE_01',
 'SEMFLU_01',
 'DSPAN_01',
 'CODING_01',
 'LRECALL_01',
 'LRECOG_01',
 'SRECALL_01',
 'MFIRECAL_01']

dtype('float64')

### Blood data processing

In [10]:
# Load blood data
blood_data = pd.read_excel(
    "/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/O-Neuro_SASP_1Notes_2Round_Sep2023.xlsx",
    sheet_name="SASP_O-Neuro_2Round",
    engine="openpyxl"
)

# Select and rename columns
blood_data = blood_data[[
    'Sample ID.1', 'Date', 'IL-6', 'gp130', 'IL-8/CXCL8',
    'uPAR', 'MIF', 'CCL2/JE/MCP-1', 'Osteoprotegerin/TNFRSF11B',
    'IL-1 beta/IL-1F2', 'CCL20/MIP-3 alpha', 'CCL3/MIP-1 alpha',
    'CCL4/MIP-1 beta', 'CCL13/MCP-4', 'GM-CSF', 'ICAM-1/CD54',
    'TNF RII/TNFRSF1B', 'TNF RI/TNFRSF1A', 'PIGF',
    'CXCL1/GRO alpha/KC/CINC-1', 'IGFBP-2', 'TIMP-1', 'IGFBP-6', 'Angiogenin'
]]

blood_data = blood_data.rename(columns={"Sample ID.1": "record_id", "Date": "blood_date"})

# Ensure blood_date is datetime
blood_data["blood_date"] = pd.to_datetime(blood_data["blood_date"], errors="coerce")

# Keep earliest sample per participant
blood_data = blood_data.sort_values(by="blood_date").drop_duplicates(subset=["record_id"], keep="first").reset_index(drop=True)

# export to csv
blood_data.to_csv("/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/processed/baseline_blood.csv", index=False)


In [14]:
for col in blood_data.columns:
    print(f"{col}: {blood_data[col].dtype}")


record_id: object
blood_date: datetime64[ns]
IL-6: float64
gp130: float64
IL-8/CXCL8: float64
uPAR: float64
MIF: float64
CCL2/JE/MCP-1: float64
Osteoprotegerin/TNFRSF11B: float64
IL-1 beta/IL-1F2: float64
CCL20/MIP-3 alpha: float64
CCL3/MIP-1 alpha: float64
CCL4/MIP-1 beta: float64
CCL13/MCP-4: float64
GM-CSF: float64
ICAM-1/CD54: float64
TNF RII/TNFRSF1B: float64
TNF RI/TNFRSF1A: float64
PIGF: float64
CXCL1/GRO alpha/KC/CINC-1: float64
IGFBP-2: float64
TIMP-1: float64
IGFBP-6: float64
Angiogenin: float64


### Clinical data processing

#### MADRS

In [16]:
### MADRS neuro

MADRS_OPTN = pd.read_excel('/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/OPT N MADRS Arms6-8 Feb2025.xlsx', engine='openpyxl')

# select relevant columns
MADRS_OPTN = MADRS_OPTN[["record_id", 'redcap_event_name', 'madrs_date', 'madrs_tot_scr']]

## filter out rows that are not baseline and week10
MADRS_OPTN = MADRS_OPTN[MADRS_OPTN['redcap_event_name'].isin(['baseline_arm_7', 'week_10end_arm_7','baseline_arm_8','week_10end_arm_8'  ])]

# for rows that are baseline, filter them into a new df caleld MADRS_OPTN_baseline, rename madras_date to baseline_madrs_date and madrs_tot_scr to baseline_madras
MADRS_OPTN_baseline = MADRS_OPTN[MADRS_OPTN['redcap_event_name'].isin(['baseline_arm_7', 'baseline_arm_8' ])]
MADRS_OPTN_baseline = MADRS_OPTN_baseline.rename(columns={"madrs_date": "baseline_madrs_date", "madrs_tot_scr": "baseline_madrs"})
MADRS_OPTN_baseline = MADRS_OPTN_baseline[["record_id", "baseline_madrs_date", "baseline_madrs"]]

# for rows that are week10, filter them into a new df caleld MADRS_OPTN_week10, rename madras_date to week10_madrs_date and madrs_tot_scr to week10_madrs
MADRS_OPTN_week10 = MADRS_OPTN[MADRS_OPTN['redcap_event_name'].isin(['week_10end_arm_7','week_10end_arm_8' ])]
MADRS_OPTN_week10 = MADRS_OPTN_week10.rename(columns={"madrs_date": "week10_madrs_date", "madrs_tot_scr": "week10_madrs"})
MADRS_OPTN_week10 = MADRS_OPTN_week10[["record_id", "week10_madrs_date", "week10_madrs"]]

# left join MADRS_OPTN_baseline and MADRS_OPTN_week10 on record_id into df called MADRS_OPTN
MADRS_OPTN = pd.merge(MADRS_OPTN_baseline, MADRS_OPTN_week10, on="record_id", how="left")

### MADRS OPT parent study

MADRS_OPT = pd.read_excel('/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/MADRS_PHQ-9_Fall_2.25.25.xlsx', skiprows=1, engine='openpyxl', sheet_name='3073_ALL_MADRS')

# remove record_ids that begin with "CU"
#MADRS_OPT = MADRS_OPT[~MADRS_OPT['record_id'].str.startswith("CU")]

MADRS_OPT = MADRS_OPT[["record_id", 'redcap_event_name', 'madrs_date', 'madrs_tot_scr']]

## filter out rows that are not baseline and week10
MADRS_OPT = MADRS_OPT[MADRS_OPT['redcap_event_name'].isin(['baseline_arm_1', 'step_1_week_10_end_arm_2' ])]


# for rows that are baseline, filter them into a new df caleld MADRS_OPTN_baseline, rename madras_date to baseline_madrs_date and madrs_tot_scr to baseline_madras
MADRS_OPT_baseline = MADRS_OPT[MADRS_OPT['redcap_event_name'].isin(['baseline_arm_1'])]
MADRS_OPT_baseline = MADRS_OPT_baseline.rename(columns={"madrs_date": "baseline_madrs_date", "madrs_tot_scr": "baseline_madrs"})
MADRS_OPT_baseline = MADRS_OPT_baseline[["record_id", "baseline_madrs_date", "baseline_madrs"]]

# for rows that are week10, filter them into a new df caleld MADRS_OPTN_week10, rename madras_date to week10_madrs_date and madrs_tot_scr to week10_madrs
MADRS_OPT_week10 = MADRS_OPT[MADRS_OPT['redcap_event_name'].isin(['step_1_week_10_end_arm_2'])]
MADRS_OPT_week10 = MADRS_OPT_week10.rename(columns={"madrs_date": "week10_madrs_date", "madrs_tot_scr": "week10_madrs"})
MADRS_OPT_week10 = MADRS_OPT_week10[["record_id", "week10_madrs_date", "week10_madrs"]]


# left join MADRS_OPT_baseline and MADRS_OPT_week10 on record_id into df called MADRS_OPT
MADRS_OPT = pd.merge(MADRS_OPT_baseline, MADRS_OPT_week10, on="record_id", how="left")

# stack MADRS_OPTN and MADRS_OPT into a new df called MADRS
MADRS = pd.concat([MADRS_OPTN, MADRS_OPT], ignore_index=True)

MADRS['record_id'] = MADRS['record_id'].str.upper()

# drop duplicate record_ids, keeping the first occurrence
MADRS = MADRS.drop_duplicates(subset=['record_id'], keep='first')

# export to csv to processed folder
MADRS.to_csv("/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/processed/OPT_madrs.csv", index=False)


In [17]:
for col in MADRS.columns:
    print(f"{col}: {MADRS[col].dtype}")


record_id: object
baseline_madrs_date: datetime64[ns]
baseline_madrs: int64
week10_madrs_date: datetime64[ns]
week10_madrs: float64


#### PHQ9

In [18]:
### PHQ9 neuro

PHQ9_OPTN = pd.read_excel('/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/OPT N PHQ9 Arms6-9 Feb2025.xlsx', engine='openpyxl')

PHQ9_OPTN = PHQ9_OPTN[PHQ9_OPTN['redcap_event_name'].isin([
    'baseline_arm_7',
    'week_2_arm_7',
    'week_4_arm_7',
    'week_6_arm_7', 
    'week_8_arm_7',
    'week_10end_arm_7',
    'baseline_arm_8',
    'week_2_arm_8',
    'week_4_arm_8',
    'week_6_arm_8',
    'week_8_arm_8',
    'week_10end_arm_8'
])]



# Baseline
PHQ9_OPTN_baseline = PHQ9_OPTN[PHQ9_OPTN['redcap_event_name'].isin(['baseline_arm_7', 'baseline_arm_8'])]
PHQ9_OPTN_baseline = PHQ9_OPTN_baseline.rename(columns={"phq9_date": "baseline_phq9_date", "phq9_total": "baseline_phq9"})
PHQ9_OPTN_baseline = PHQ9_OPTN_baseline[["record_id", "baseline_phq9_date", "baseline_phq9"]]


# Week 2
PHQ9_OPTN_week2 = PHQ9_OPTN[PHQ9_OPTN['redcap_event_name'].isin(['week_2_arm_7', 'week_2_arm_8'])]
PHQ9_OPTN_week2 = PHQ9_OPTN_week2.rename(columns={"phq9_date": "week2_phq9_date", "phq9_total": "week2_phq9"})
PHQ9_OPTN_week2 = PHQ9_OPTN_week2[["record_id", "week2_phq9_date", "week2_phq9"]]

# Week 4
PHQ9_OPTN_week4 = PHQ9_OPTN[PHQ9_OPTN['redcap_event_name'].isin(['week_4_arm_7', 'week_4_arm_8'])]
PHQ9_OPTN_week4 = PHQ9_OPTN_week4.rename(columns={"phq9_date": "week4_phq9_date", "phq9_total": "week4_phq9"})
PHQ9_OPTN_week4 = PHQ9_OPTN_week4[["record_id", "week4_phq9_date", "week4_phq9"]]

# Week 6
PHQ9_OPTN_week6 = PHQ9_OPTN[PHQ9_OPTN['redcap_event_name'].isin(['week_6_arm_7', 'week_6_arm_8'])]
PHQ9_OPTN_week6 = PHQ9_OPTN_week6.rename(columns={"phq9_date": "week6_phq9_date", "phq9_total": "week6_phq9"})
PHQ9_OPTN_week6 = PHQ9_OPTN_week6[["record_id", "week6_phq9_date", "week6_phq9"]]

# Week 8
PHQ9_OPTN_week8 = PHQ9_OPTN[PHQ9_OPTN['redcap_event_name'].isin(['week_8_arm_7', 'week_8_arm_8'])]
PHQ9_OPTN_week8 = PHQ9_OPTN_week8.rename(columns={"phq9_date": "week8_phq9_date", "phq9_total": "week8_phq9"})
PHQ9_OPTN_week8 = PHQ9_OPTN_week8[["record_id", "week8_phq9_date", "week8_phq9"]]

# Week 10
PHQ9_OPTN_week10 = PHQ9_OPTN[PHQ9_OPTN['redcap_event_name'].isin(['week_10end_arm_7', 'week_10end_arm_8'])]
PHQ9_OPTN_week10 = PHQ9_OPTN_week10.rename(columns={"phq9_date": "week10_phq9_date", "phq9_total": "week10_phq9"})
PHQ9_OPTN_week10 = PHQ9_OPTN_week10[["record_id", "week10_phq9_date", "week10_phq9"]]

# Merge all timepoints on record_id
PHQ9_OPTN = PHQ9_OPTN_baseline
for df in [PHQ9_OPTN_week2, PHQ9_OPTN_week4, PHQ9_OPTN_week6, PHQ9_OPTN_week8, PHQ9_OPTN_week10]:
    PHQ9_OPTN = pd.merge(PHQ9_OPTN, df, on="record_id", how="left")


### PHQ9 OPT parent study
PHQ9_OPT = pd.read_excel('/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/MADRS_PHQ-9_Fall_2.25.25.xlsx', skiprows=0, engine='openpyxl', sheet_name='6237 All Item PHQ-9_7.27.23')

# rename columns
PHQ9_OPT = PHQ9_OPT.rename(columns={
    'Record ID': 'record_id',
    'Event Name': 'redcap_event_name',
    'Corrected -Date completing PhQ-9': 'phq9_date',
    'phq9_q1': 'phq9_1',
    'phq9_q2': 'phq9_2',
    'phq9_q3': 'phq9_3',
    'phq9_q4': 'phq9_4',
    'phq9_q5': 'phq9_5',
    'phq9_q6': 'phq9_6',
    'phq9_q7': 'phq9_7',
    'phq9_q8': 'phq9_8',
    'phq9_q9': 'phq9_9',
    'Corrected phq9_tot_scr': 'phq9_total'})

# remove record_ids that begin with "CU" and make all record_id uppercase
# PHQ9_OPT = PHQ9_OPT[~PHQ9_OPT['record_id'].str.startswith("CU")]


# Baseline
PHQ9_OPT_baseline = PHQ9_OPT[PHQ9_OPT['redcap_event_name'].isin(['baseline_arm_1'])]
PHQ9_OPT_baseline = PHQ9_OPT_baseline.rename(columns={"phq9_date": "baseline_phq9_date", "phq9_total": "baseline_phq9"})
PHQ9_OPT_baseline = PHQ9_OPT_baseline[["record_id", "baseline_phq9_date", "baseline_phq9"]]


# Week 2
PHQ9_OPT_week2 = PHQ9_OPT[PHQ9_OPT['redcap_event_name'].isin(['step_1_week_2_arm_2'])]
PHQ9_OPT_week2 = PHQ9_OPT_week2.rename(columns={"phq9_date": "week2_phq9_date", "phq9_total": "week2_phq9"})
PHQ9_OPT_week2 = PHQ9_OPT_week2[["record_id", "week2_phq9_date", "week2_phq9"]]


# Week 4
PHQ9_OPT_week4 = PHQ9_OPT[PHQ9_OPT['redcap_event_name'].isin(['step_1_week_4_arm_2'])]
PHQ9_OPT_week4 = PHQ9_OPT_week4.rename(columns={"phq9_date": "week4_phq9_date", "phq9_total": "week4_phq9"})
PHQ9_OPT_week4 = PHQ9_OPT_week4[["record_id", "week4_phq9_date", "week4_phq9"]]

# Week 6
PHQ9_OPT_week6 = PHQ9_OPT[PHQ9_OPT['redcap_event_name'].isin(['step_1_week_6_arm_2'])]
PHQ9_OPT_week6 = PHQ9_OPT_week6.rename(columns={"phq9_date": "week6_phq9_date", "phq9_total": "week6_phq9"})
PHQ9_OPT_week6 = PHQ9_OPT_week6[["record_id", "week6_phq9_date", "week6_phq9"]]

# Week 8
PHQ9_OPT_week8 = PHQ9_OPT[PHQ9_OPT['redcap_event_name'].isin(['step_1_week_8_arm_2'])]
PHQ9_OPT_week8 = PHQ9_OPT_week8.rename(columns={"phq9_date": "week8_phq9_date", "phq9_total": "week8_phq9"})
PHQ9_OPT_week8 = PHQ9_OPT_week8[["record_id", "week8_phq9_date", "week8_phq9"]]

# Week 10
PHQ9_OPT_week10 = PHQ9_OPT[PHQ9_OPT['redcap_event_name'].isin(['step_1_week_10_end_arm_2'])]
PHQ9_OPT_week10 = PHQ9_OPT_week10.rename(columns={"phq9_date": "week10_phq9_date", "phq9_total": "week10_phq9"})
PHQ9_OPT_week10 = PHQ9_OPT_week10[["record_id", "week10_phq9_date", "week10_phq9"]]

# Merge all timepoints on record_id
PHQ9_OPT = PHQ9_OPT_baseline
for df in [PHQ9_OPT_week2, PHQ9_OPT_week4, PHQ9_OPT_week6, PHQ9_OPT_week8, PHQ9_OPT_week10]:
    PHQ9_OPT = pd.merge(PHQ9_OPT, df, on="record_id", how="left")

# stack PHQ9_OPTN and PHQ9_OPT into a new df called PHQ9
PHQ9 = pd.concat([PHQ9_OPTN, PHQ9_OPT], ignore_index=True)
PHQ9['record_id'] = PHQ9['record_id'].str.upper()

# drop duplicate record_ids, keeping the first occurrence
PHQ9 = PHQ9.drop_duplicates(subset=['record_id'], keep='first')

# export to csv to processed folder
PHQ9.to_csv("/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/processed/OPT_phq9.csv", index=False)


In [19]:
for col in PHQ9.columns:
    print(f"{col}: {PHQ9[col].dtype}")


record_id: object
baseline_phq9_date: datetime64[ns]
baseline_phq9: float64
week2_phq9_date: datetime64[ns]
week2_phq9: float64
week4_phq9_date: datetime64[ns]
week4_phq9: float64
week6_phq9_date: datetime64[ns]
week6_phq9: float64
week8_phq9_date: datetime64[ns]
week8_phq9: float64
week10_phq9_date: datetime64[ns]
week10_phq9: float64


### Demographics and randomization 

In [26]:
## Pull in randomization excel

OPT_randomization = clean_up_df(pd.read_excel('/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/OPT N Randomization Groups Dates 20250401.xlsx', engine='openpyxl'), 'record_id')

# select relevant columns
OPT_randomization = OPT_randomization[['record_id','step1_rand_group', 'step1_rand_date']]

## drop rows where step1_rand_group is "Not Applicable"
#OPT_randomization = OPT_randomization[OPT_randomization['step1_rand_group'] != "Not Applicable"]


###  1. Load Arm 6 demographics (OPT_neuro)

# load up the columns id, site, gender, race, ethnicity, edu_lvl, age from core variabels csv as OPT_demographics
OPT_demographics = pd.read_csv(
    '/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/CoreVariable_4.17.25.csv',
    usecols=['id', 'site', 'gender', 'race', 'ethnicity', 'edu_lvl', 'age', 'bmi'],
    skiprows=[1]
)
# rename id to record_id and make all record_id uppercase
OPT_demographics = OPT_demographics.rename(columns={"id": "record_id"})


###  2. Load Arm 6 demographics (OPT_neuro)
OPT_neuro = pd.read_excel(
    '/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/OPT N Demo Arm 6.xlsx', engine='openpyxl'
)
OPT_neuro['record_id'] = OPT_neuro['record_id'].str.upper()

# Rename columns
OPT_neuro = OPT_neuro.rename(columns={
    'demo_age': 'age',
    'demo_sex': 'gender',
    'demo_race': 'race',
    'demo_ethnicity': 'ethnicity',
    'demo_edu': 'edu_lvl'
})
OPT_neuro = OPT_neuro[['record_id', 'age', 'gender', 'race', 'ethnicity', 'edu_lvl']]

###  3. Load age sheet (OPT_neuro_age)
OPT_neuro_age = pd.read_excel(
    '/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/OPT N Age Arms 7-8 20250404.xlsx', engine='openpyxl'
)
OPT_neuro_age['record_id'] = OPT_neuro_age['record_id'].str.upper()
OPT_neuro_age['age'] = OPT_neuro_age[['Age_Step1', 'Age_Step2', 'Age_Consent']].bfill(axis=1).iloc[:, 0]
OPT_neuro_age = OPT_neuro_age[['record_id', 'age']]  # Drop unused columns

# Merge age into OPT_neuro (outer merge to preserve all unique IDs)
OPT_neuro = pd.merge(
    OPT_neuro,
    OPT_neuro_age,
    on='record_id',
    how='outer',
    suffixes=('', '_age')
)
OPT_neuro['age'] = OPT_neuro['age'].combine_first(OPT_neuro['age_age'])
OPT_neuro = OPT_neuro.drop(columns=['age_age'])

###  4. Load Arm 7–8 demographic supplement (OPT_neuro_demo)
OPT_neuro_demo = pd.read_excel(
    '/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/OPT N Demo Arm 7-8.xlsx', engine='openpyxl'
)
OPT_neuro_demo['record_id'] = OPT_neuro_demo['record_id'].str.upper()
OPT_neuro_demo = OPT_neuro_demo.rename(columns={
    'demo_age': 'age',
    'demo_sex': 'gender',
    'demo_race': 'race',
    'demo_ethnicity': 'ethnicity'
})
OPT_neuro_demo = OPT_neuro_demo[['record_id', 'gender', 'race', 'ethnicity']]

# Merge additional demographic info into OPT_neuro
OPT_neuro = pd.merge(
    OPT_neuro,
    OPT_neuro_demo,
    on='record_id',
    how='outer',
    suffixes=('', '_supp')
)
for col in ['gender', 'race', 'ethnicity']:
    OPT_neuro[col] = OPT_neuro[col].combine_first(OPT_neuro[f'{col}_supp'])
    OPT_neuro = OPT_neuro.drop(columns=[f'{col}_supp'])

### 5. Merge OPT_neuro into OPT_demographics
OPT_demographics = pd.merge(
    OPT_demographics,
    OPT_neuro,
    on='record_id',
    how='outer',
    suffixes=('', '_neuro')
)

# fill in non-null values from neuro sheet for missing entries
for col in ['age', 'gender', 'race', 'ethnicity', 'edu_lvl']:
    OPT_demographics[col] = OPT_demographics[col].combine_first(OPT_demographics[f'{col}_neuro'])
    OPT_demographics = OPT_demographics.drop(columns=[f'{col}_neuro'])


# get BMI for OPT_neuro

OPT_neuro_bmi = pd.read_excel(
    '/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/OPT N Height Weight Arm 8 20250401.xlsx',
    engine='openpyxl',
    usecols=['record_id', 'bmi']
)

# Ensure record_id consistency
OPT_neuro_bmi['record_id'] = OPT_neuro_bmi['record_id'].str.upper()

# Merge BMI into demographics
OPT_demographics = pd.merge(
    OPT_demographics,
    OPT_neuro_bmi,
    on='record_id',
    how='left'
)

if 'bmi_x' in OPT_demographics.columns and 'bmi_y' in OPT_demographics.columns:
    OPT_demographics['bmi'] = OPT_demographics['bmi_y'].combine_first(OPT_demographics['bmi_x'])
    OPT_demographics = OPT_demographics.drop(columns=['bmi_x', 'bmi_y'])

# merge randomization into demographics

OPT_demographics = pd.merge(
    OPT_demographics,
    OPT_randomization,
    on='record_id',
    how='left'
)


### 7. Final cleanup
OPT_demographics = OPT_demographics.drop_duplicates(subset='record_id', keep='first')
OPT_demographics = OPT_demographics.reset_index(drop=True)



OPT_demographics['gender'] = (
    OPT_demographics['gender']
    .astype(str)                       # Ensure it's string
    .str.extract(r'(\d+)')[0]          # Extract numeric part before any dot
    .map(gender_code)                  # Map using your dictionary
    .fillna(OPT_demographics['gender'])  # Keep original if not in mapping
)

OPT_demographics['race'] = (
    OPT_demographics['race']
    .astype(str)                       # Ensure it's string
    .str.extract(r'(\d+)')[0]          # Extract numeric part before any dot
    .map(race_code)                    # Map using your dictionary
    .fillna(OPT_demographics['race'])  # Keep original if not in mapping
)

OPT_demographics['ethnicity'] = (
    OPT_demographics['ethnicity']
    .astype(str)
    .str.extract(r'(\d+)')[0]
    .map(ethnicity_code)
    .fillna(OPT_demographics['ethnicity'])
)

# map missing code to entire df
OPT_demographics = OPT_demographics.replace(missing_code)

# replace values in step1_rand_group using the dictionary --- first make it a string since its integer 
OPT_demographics['step1_rand_group'] = (
    OPT_demographics['step1_rand_group']
    .astype(str)
    .replace(randomization)
)


# clean up the columns
OPT_demographics['edu_lvl'] = clean_numeric_column(OPT_demographics['edu_lvl'])
OPT_demographics['age'] = clean_numeric_column(OPT_demographics['age'])

# correct bmi to 2 significant figures
OPT_demographics['bmi'] = (
    OPT_demographics['bmi']
    .astype(str)
    .str.extract(r'(\d+\.\d{0,2})')[0]  # Extract up to 2 decimal places
    .astype(float)                      # Convert back to float
)
# set continuous columns to dtype float

OPT_demographics['age'] = pd.to_numeric(OPT_demographics['age'], errors='coerce')
OPT_demographics['bmi'] = pd.to_numeric(OPT_demographics['bmi'], errors='coerce')
OPT_demographics['edu_lvl'] = pd.to_numeric(OPT_demographics['edu_lvl'], errors='coerce')


# export to csv
OPT_demographics.to_csv("/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/processed/OPT_demographics.csv", index=False)

Total missing codes replaced across DataFrame: 1269
Column 'record_id': 0 empty values removed


In [27]:
for col in OPT_demographics.columns:
    print(f"{col}: {OPT_demographics[col].dtype}")


record_id: object
site: object
gender: object
race: object
ethnicity: object
edu_lvl: float64
age: float64
bmi: float64
step1_rand_group: object
step1_rand_date: object


### Medication adherence

In [29]:
# load up decision support form from OPT Neuro

OPT_neuro_decision_support = pd.read_excel('/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/OPT N Decision Support Form Arm 7-8.xlsx', engine='openpyxl')
OPT_neuro_decision_support['record_id'] = OPT_neuro_decision_support['record_id'].str.upper()

# filter out participants that end with arm_7 from redcaop_event_name (basically only keep the step1 participants)
OPT_neuro_decision_support = OPT_neuro_decision_support[~OPT_neuro_decision_support['redcap_event_name'].str.endswith('arm_7')]


# select relevant columns

OPT_neuro_decision_support = OPT_neuro_decision_support[["record_id", "redcap_event_name", "decision_date","adherence_3_med1", 'type_in_rec_med1', 'adherence_3_freq1',"adherence_3_med2",'type_in_rec_med2','adherence_3_freq2', "adherence_4_name"]]

# filter out rows where redcap_event_name does not begin with week
OPT_neuro_decision_support = OPT_neuro_decision_support[OPT_neuro_decision_support['redcap_event_name'].str.startswith('week')]

# Map event labels for clarity
OPT_neuro_decision_support['event_label'] = OPT_neuro_decision_support['redcap_event_name'].map({
    'week_2_arm_8': 'week2',
    'week_4_arm_8': 'week4',
    'week_6_arm_8': 'week6',
    'week_8_arm_8': 'week8',
    'week_10end_arm_8': 'week10'
})

# Select and rename columns before pivot
OPT_neuro_decision_support = OPT_neuro_decision_support[[
    'record_id', 'event_label', 'decision_date', 'type_in_rec_med1', 'adherence_3_freq1', 'type_in_rec_med2', 'adherence_3_freq2'
]]

# Melt to long for easier reformatting
long_df = OPT_neuro_decision_support.melt(
    id_vars=['record_id', 'event_label'],
    var_name='measure',
    value_name='value'
)

# Create unified column names like week2_date, week2_med1, etc.
long_df['wide_col'] = long_df['event_label'] + '_' + long_df['measure'].str.replace('decision_date', 'date') \
                                                           .str.replace('type_in_rec_med1', 'med1') \
                                                           .str.replace('adherence_3_freq1', 'freq1') \
                                                           .str.replace('type_in_rec_med2', 'med2') \
                                                           .str.replace('adherence_3_freq2', 'freq2')

# Pivot to wide format
OPT_neuro_decision_support = long_df.pivot(index='record_id', columns='wide_col', values='value').reset_index()

# define timepoints to reorder it nicely
timepoints = ['week2', 'week4', 'week6', 'week8', 'week10']
variables = ['date', 'med1', 'freq1', 'med2', 'freq2']

# loop through lists to make the desired order
desired_order = ['record_id'] + [f'{tp}_{var}' for tp in timepoints for var in variables]

# reorder the dataframe
OPT_neuro_decision_support = OPT_neuro_decision_support[desired_order]

# map frequency dictionary to the frequency columns
## freq1
for tp in timepoints:
    freq_col = f'{tp}_freq1'
    if freq_col in OPT_neuro_decision_support.columns:
        OPT_neuro_decision_support[freq_col] = (
            OPT_neuro_decision_support[freq_col]
            .astype(str)
            .str.extract(r'(\d+)')[0]
            .map(frequency_code)
            .fillna(OPT_neuro_decision_support[freq_col])
        )
## freq2
for tp in timepoints:
    freq_col = f'{tp}_freq2'
    if freq_col in OPT_neuro_decision_support.columns:
        OPT_neuro_decision_support[freq_col] = (
            OPT_neuro_decision_support[freq_col]
            .astype(str)
            .str.extract(r'(\d+)')[0]
            .map(frequency_code)
            .fillna(OPT_neuro_decision_support[freq_col])
        )




# load up OPT parent adherence decision support form

OPT_parent_decision_support = pd.read_csv('/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/Adherence-SideEffect-Yes-No_1.24.25.csv', skiprows=1)
OPT_parent_decision_support['record_id'] = OPT_parent_decision_support['record_id'].str.upper()

# standardize date column 
OPT_parent_decision_support['decision_date'] = standardize_dates(OPT_parent_decision_support['decision_date'])

# filter out participants that end with arm_7 from redcaop_event_name (basically only keep the step1 participants)
OPT_parent_decision_support = OPT_parent_decision_support[
    ~OPT_parent_decision_support['redcap_event_name'].str.startswith(('Step 2', 'Month', 'Extra', 'Week 0'))
]

# select relevant columns

OPT_parent_decision_support = OPT_parent_decision_support[["record_id", "redcap_event_name", "decision_date","adherence_3_med1", 'type_in_rec_med1', 'adherence_3_freq1',"adherence_3_med2",'type_in_rec_med2','adherence_3_freq2', "adherence_4_name"]]

# Map event labels for clarity
OPT_parent_decision_support['event_label'] = OPT_parent_decision_support['redcap_event_name'].map({
    'Step 1 Week 2 (Arm 2: Step 1 Acute)': 'week2',
    'Step 1 Week 4 (Arm 2: Step 1 Acute)': 'week4',
    'Step 1 Week 6 (Arm 2: Step 1 Acute)': 'week6',
    'Step 1 Week 8 (Arm 2: Step 1 Acute)': 'week8',
    'Step 1 Week 10/ End (Arm 2: Step 1 Acute)': 'week10',
    'Step 1 Extra DecisionSupport 1 (Arm 2: Step 1 Acute)': 'extra1',
    'Step 1 Extra DecisionSupport 2 (Arm 2: Step 1 Acute)': 'extra2',
    'Step 1 Extra DecisionSupport 3 (Arm 2: Step 1 Acute)': 'extra3'
})

# Select and rename columns before pivot
OPT_parent_decision_support = OPT_parent_decision_support[[
    'record_id', 'event_label', 'decision_date', 'adherence_3_med1', 'adherence_3_freq1', 'adherence_3_med2', 'adherence_3_freq2'
]]

# Melt to long for easier reformatting
long_parent_df = OPT_parent_decision_support.melt(
    id_vars=['record_id', 'event_label'],
    var_name='measure',
    value_name='value'
)

# Create unified column names like week2_date, week2_med1, etc.
long_parent_df['wide_col'] = long_parent_df['event_label'] + '_' + long_parent_df['measure'].str.replace('decision_date', 'date') \
                                                           .str.replace('adherence_3_med1', 'med1') \
                                                           .str.replace('adherence_3_freq1', 'freq1') \
                                                           .str.replace('adherence_3_med2', 'med2') \
                                                           .str.replace('adherence_3_freq2', 'freq2')


# drop rows where event_label begins with extra
long_parent_df = long_parent_df[~long_parent_df['event_label'].str.startswith('extra')]

# Pivot to wide format
OPT_parent_decision_support = long_parent_df.pivot(index='record_id', columns='wide_col', values='value').reset_index()

# define timepoints to reorder it nicely
timepoints = ['week2', 'week4', 'week6', 'week8', 'week10']
variables = ['date', 'med1', 'freq1', 'med2', 'freq2']

# loop through lists to make the desired order
desired_order = ['record_id'] + [f'{tp}_{var}' for tp in timepoints for var in variables]

# reorder the dataframe
OPT_parent_decision_support = OPT_parent_decision_support[desired_order]

# map frequency dictionary to the frequency columns
## freq1
for tp in timepoints:
    freq_col = f'{tp}_freq1'
    if freq_col in OPT_parent_decision_support.columns:
        OPT_parent_decision_support[freq_col] = (
            OPT_parent_decision_support[freq_col]
            .astype(str)
            .str.extract(r'(\d+)')[0]
            .map(frequency_code)
            .fillna(OPT_parent_decision_support[freq_col])
        )
## freq2
for tp in timepoints:
    freq_col = f'{tp}_freq2'
    if freq_col in OPT_parent_decision_support.columns:
        OPT_parent_decision_support[freq_col] = (
            OPT_parent_decision_support[freq_col]
            .astype(str)
            .str.extract(r'(\d+)')[0]
            .map(frequency_code)
            .fillna(OPT_parent_decision_support[freq_col])
        )


# stack OPT_neuro_decision_support and OPT_parent_decision_support into a new df called decision_support
OPT_decision_support = pd.concat([OPT_neuro_decision_support, OPT_parent_decision_support], ignore_index=True)

# sort the rows by record_id
OPT_decision_support = OPT_decision_support.sort_values(by='record_id').reset_index(drop=True)

## some med names are not caps and are all lower case, whereas some that are the same name are upper. we will standardize them to be all upper

# Identify all columns that are med1 or med2
med_cols = [col for col in OPT_decision_support.columns if '_med1' in col or '_med2' in col]
# Apply .str.upper() to all med columns

for col in med_cols:
    # Apply upper() only on non-NaNs
    OPT_decision_support[col] = OPT_decision_support[col].where(
        OPT_decision_support[col].isna(),  # keep NaNs as-is
        OPT_decision_support[col].astype(str).str.upper()  # apply upper() to non-NaNs only
    )

    # Apply mapping, preserving NaNs
    OPT_decision_support[col] = OPT_decision_support[col].map(medication_map).fillna(OPT_decision_support[col])


# apply missing code across the entire df
OPT_decision_support = OPT_decision_support.replace(missing_code)

# export to csv
OPT_decision_support.to_csv("/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/processed/OPT_decision_support.csv", index=False)

In [30]:
for col in OPT_decision_support.columns:
    print(f"{col}: {OPT_decision_support[col].dtype}")


record_id: object
week2_date: datetime64[ns]
week2_med1: object
week2_freq1: object
week2_med2: object
week2_freq2: object
week4_date: datetime64[ns]
week4_med1: object
week4_freq1: object
week4_med2: object
week4_freq2: object
week6_date: datetime64[ns]
week6_med1: object
week6_freq1: object
week6_med2: object
week6_freq2: object
week8_date: datetime64[ns]
week8_med1: object
week8_freq1: object
week8_med2: object
week8_freq2: object
week10_date: datetime64[ns]
week10_med1: object
week10_freq1: object
week10_med2: object
week10_freq2: object


### NIH toolbox -  motor and cog

In [31]:
### NIH toolbox (COG)


OPT_nih_toolbox = pd.read_csv('/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/NIHTB_Cog_step1_3.28.25.csv')

# rename column id to record_id
OPT_nih_toolbox = OPT_nih_toolbox.rename(columns={"id": "record_id", "vdate" : "nih_date"})

# fix record_id
OPT_nih_toolbox['record_id'] = OPT_nih_toolbox['record_id'].str.upper()

# standardize date column vdate
OPT_nih_toolbox['nih_date'] = standardize_dates(OPT_nih_toolbox['nih_date'])

# rename values in time column if OPTBL set to baseline if OPTS1End to week10
OPT_nih_toolbox['time'] = OPT_nih_toolbox['time'].replace({'OPTBL': 'baseline', 'OPTS1End': 'week10'})

# filter out rows that are not baseline and week10
OPT_nih_toolbox = OPT_nih_toolbox[OPT_nih_toolbox['time'].isin(['baseline', 'week10'])]

# coalesce fcc and fcc_imputed columns
OPT_nih_toolbox['fcc'] = OPT_nih_toolbox['fcc'].combine_first(OPT_nih_toolbox['fcc_imputed'])

# select relevant columns
OPT_nih_toolbox = OPT_nih_toolbox[["record_id", 'nih_date', 'time', 'fcc',  'dccs',
       'flanker', 'listSort', 'pattComp', 'psm', ]]

# pivot the time column to make it wide
OPT_nih_toolbox = OPT_nih_toolbox.pivot(index='record_id', columns='time', values=['nih_date', 'fcc', 'dccs', 'flanker', 'listSort', 'pattComp', 'psm']).reset_index()
# flatten the multi-level columns
OPT_nih_toolbox.columns = ['_'.join(col).strip() for col in OPT_nih_toolbox.columns.values]


OPT_nih_toolbox.to_csv("/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/processed/OPT_nih_toolbox_cog.csv", index=False)
print("NIH toolbox cog data saved to /external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/processed/OPT_nih_toolbox_cog.csv")

### NIH toolbox (motor)

OPT_nih_toolbox_motor = pd.read_excel('/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/NIHTB_Motor_4.16.25_Yi.xlsx',
    sheet_name="s1_motor_final_041325", ## step 1
    engine="openpyxl"
)

# rename columns id to record_id and vdate to nih_date
OPT_nih_toolbox_motor = OPT_nih_toolbox_motor.rename(columns={"id": "record_id", "vdate" : "nih_date"})

# fix record_id
OPT_nih_toolbox_motor['record_id'] = OPT_nih_toolbox_motor['record_id'].str.upper()

# standardize date column vdate
OPT_nih_toolbox_motor['nih_date'] = standardize_dates(OPT_nih_toolbox_motor['nih_date'])

# rename values in time column if OPTBL set to baseline if OPTS1End to week10
OPT_nih_toolbox_motor['time'] = OPT_nih_toolbox_motor['time'].replace({'OPTBL': 'baseline', 'OPTS1End': 'week10'})

# filter out rows that are not baseline and week10
OPT_nih_toolbox_motor = OPT_nih_toolbox_motor[OPT_nih_toolbox_motor['time'].isin(['baseline', 'week10'])]

# select relevant columns
OPT_nih_toolbox_motor = OPT_nih_toolbox_motor[["record_id",'nih_date', 'time', 'task', 'raw_scr',
       'comp_scr', 'unc_scr', 'unc_scr_dom', 'unc_scr_ndom', 'dom_scr',
       'ndom_scr']]


OPT_nih_toolbox_motor = OPT_nih_toolbox_motor.pivot(index='record_id', columns=['time', 'task'],  values=['raw_scr', 'comp_scr', 'unc_scr', 'unc_scr_dom', 'unc_scr_ndom', 'dom_scr', 'ndom_scr']).reset_index()

OPT_nih_toolbox_motor.columns = [
    'record_id' if metric == 'record_id' else f"{task}_{time}_{metric}"
    for metric, time, task in OPT_nih_toolbox_motor.columns.values
]
## drop columns that are completely null
OPT_nih_toolbox_motor = OPT_nih_toolbox_motor.dropna(axis=1, how='all')
# export to csv
OPT_nih_toolbox_motor.to_csv("/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/processed/OPT_nih_toolbox_motor.csv", index=False)
print("NIH toolbox motor data saved to /external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/processed/OPT_nih_toolbox_motor.csv")

NIH toolbox cog data saved to /external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/processed/OPT_nih_toolbox_cog.csv


NIH toolbox motor data saved to /external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/processed/OPT_nih_toolbox_motor.csv


In [32]:
for col in OPT_nih_toolbox_motor.columns:
    print(f"{col}: {OPT_nih_toolbox_motor[col].dtype}")


record_id: object
4mWalk_baseline_raw_scr: float64
4mWalk_week10_raw_scr: float64
4mWalk_baseline_comp_scr: float64
4mWalk_week10_comp_scr: float64
StandingBalance_baseline_unc_scr: float64
StandingBalance_week10_unc_scr: float64
9HolePegboard_baseline_unc_scr_dom: float64
gripStrength_baseline_unc_scr_dom: float64
9HolePegboard_week10_unc_scr_dom: float64
gripStrength_week10_unc_scr_dom: float64
9HolePegboard_baseline_unc_scr_ndom: float64
gripStrength_baseline_unc_scr_ndom: float64
9HolePegboard_week10_unc_scr_ndom: float64
gripStrength_week10_unc_scr_ndom: float64
9HolePegboard_baseline_dom_scr: float64
gripStrength_baseline_dom_scr: float64
9HolePegboard_week10_dom_scr: float64
gripStrength_week10_dom_scr: float64
9HolePegboard_baseline_ndom_scr: float64
gripStrength_baseline_ndom_scr: float64
9HolePegboard_week10_ndom_scr: float64
gripStrength_week10_ndom_scr: float64


In [ ]:
### NIH filtering -- we might need this might not -- probably move this to master_table_dashboard 


# drop rows where nih_date_week10 is null and then drop rows where nih_date_baseline is null
OPT_nih_toolbox = OPT_nih_toolbox.dropna(subset=['nih_date_week10'])
OPT_nih_toolbox = OPT_nih_toolbox.dropna(subset=['nih_date_baseline'])




## make all column headers lower case
OPT_nih_toolbox_motor.columns = OPT_nih_toolbox_motor.columns.str.lower()

## drop rows that have a single null 
OPT_nih_toolbox_motor = OPT_nih_toolbox_motor.dropna(axis=0, how='any')

### baseline date (date first dose was taken)

In [20]:
OPT_neuro_baseline_date =  pd.read_excel("/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/OPT N Participant Tracking Form Arm 7-8.xlsx",engine="openpyxl")[['record_id', 'tracking_step1_r4']].rename(columns={'tracking_step1_r4': 'baseline_date'})

OPT_parent_baseline_date = pd.read_csv("/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/Date_1stDoseStep1_5.22.25.csv", skiprows=1)[['record_id', 'tracking_step1_r4']].rename(columns={'tracking_step1_r4': 'baseline_date'})

#standerdize date

OPT_parent_baseline_date['baseline_date']=standardize_dates(OPT_parent_baseline_date['baseline_date'])

OPT_baseline_date = pd.concat([OPT_parent_baseline_date, OPT_neuro_baseline_date], ignore_index=True)
OPT_baseline_date['record_id'] = OPT_baseline_date['record_id'].str.upper()

# apply missing code to df
OPT_baseline_date = OPT_baseline_date.replace(missing_code)

# write to csv

OPT_baseline_date.to_csv("/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/processed/OPT_baseline_date.csv", index=False)

### side effects

In [47]:
OPT_parent_sideeffects = pd.read_csv('/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/Adherence-SideEffect-Yes-No_1.24.25.csv')

### medications 

In [8]:
OPT_neuro_medications = pd.read_excel('/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/OPT N Meds Arm 6-8.xlsx', engine='openpyxl')[['record_id', 
        'date_med_entered', 
       'type_in_med', 'med_freq',
       'med_freq_oth', 'med_start', 'non_psych_med_baseline___1', 
       'med_reason']]

OPT_neuro_medications['type_in_med_standardized'] = OPT_neuro_medications['type_in_med'].astype(str).str.upper().str.strip()


In [7]:
unique_meds = OPT_neuro_medications['type_in_med_standardized'].dropna().unique().tolist()


In [8]:
# list out all the unique values in the type_in_med column
OPT_neuro_medications['type_in_med'].unique()

array(['fenofibrate', 'melatonon plus', 'premarin cream', ...,
       'Verapamil', 'MULTI VITAMIN', 'EXEMESTATE'], dtype=object)

In [ ]:


def normalize_text(text):
    text = str(text).upper()
    text = re.sub(r'[^A-Z0-9\s]', '', text)     # remove punctuation
    text = re.sub(r'\s+', ' ', text).strip()    # normalize whitespace
    return text

OPT_neuro_medications['type_in_med_clean'] = OPT_neuro_medications['type_in_med'].apply(normalize_text)


known_drugs = [
    "AMLODIPINE", "SERTRALINE", "FLUOXETINE", "BUPROPION",
    "METFORMIN", "RISPERIDONE", "OMEGA 3", "MULTIVITAMIN"
]

def fuzzy_match_to_known(text, known_list=known_drugs, threshold=85):
    match, score, _ = process.extractOne(text, known_list, scorer=fuzz.token_sort_ratio)
    return match if score >= threshold else text  # fallback to original

OPT_neuro_medications['med_fuzzy'] = OPT_neuro_medications['type_in_med_clean'].apply(fuzzy_match_to_known)

import requests

def rxnorm_lookup_name(name):
    try:
        url = f"https://rxnav.nlm.nih.gov/REST/approximateTerm.json?term={name}"
        r = requests.get(url)
        candidates = r.json().get('approximateGroup', {}).get('candidate', [])
        if not candidates:
            return None, None, None
        rxcui = candidates[0]['rxcui']
        # Get canonical RxNorm name
        props_url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/properties.json"
        props = requests.get(props_url).json()
        rxnorm_name = props.get('properties', {}).get('name')

        # Get ingredient-level CUI + name
        ingr_url = f"https://rxnav.nlm.nih.gov/REST/rxcui/{rxcui}/related.json?tty=IN"
        ingr_resp = requests.get(ingr_url).json()
        concept = ingr_resp.get('relatedGroup', {}).get('conceptGroup', [{}])[0].get('conceptProperties', [{}])[0]
        ingredient_name = concept.get('name')
        ingredient_cui = concept.get('rxcui')
        return rxcui, rxnorm_name, ingredient_name or rxnorm_name
    except:
        return None, None, None

def lookup_all(med):
    rxcui, rxname, base = rxnorm_lookup_name(med)
    return pd.Series([rxcui, rxname, base])

OPT_neuro_medications[['rxcui', 'rxnorm_name', 'ingredient_name']] = OPT_neuro_medications['med_fuzzy'].apply(lookup_all)


In [15]:

OPT_neuro_medications.to_csv("/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/processed/OPT_neuro_medications.csv", index=False)


group 1: vitamins and supplmements like fish oils and melatonin
group 2: over the counter meds
group 3: prescription meds
group 4: heavy meds 

### falls

In [30]:
OPT_neuro_falls = pd.read_excel('/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/OPT N Decision Support Form Arm 7-8.xlsx' , engine='openpyxl')[['record_id', 'redcap_event_name','decision_fall_1', 'decision_fall_2___5']]

OPT_neuro_falls['record_id'] = OPT_neuro_falls['record_id'].str.upper()


# filter out rows where redcap_event_name does not begin with week and ends with arm_8
OPT_neuro_falls = OPT_neuro_falls[
    OPT_neuro_falls['redcap_event_name'].str.startswith('week') &
    OPT_neuro_falls['redcap_event_name'].str.endswith('arm_8')
]
# Map event labels for clarity
OPT_neuro_falls['redcap_event_name'] = OPT_neuro_falls['redcap_event_name'].map({
    'week_2_arm_8': 'week2',
    'week_4_arm_8': 'week4',
    'week_6_arm_8': 'week6',
    'week_8_arm_8': 'week8',
    'week_10end_arm_8': 'week10'
})

# map decision_fall_1 dictionary to df
OPT_neuro_falls['decision_fall_1'] = (
    OPT_neuro_falls['decision_fall_1']
    .map(fall_code)
)


fall_column_rename = {
    'decision_fall_1': 'number_falls',
    'decision_fall_2___5': 'fall_injury'}

OPT_neuro_falls.rename(columns=fall_column_rename, inplace=True)

#pivot the dataframe to wide format by record_id and redcap_event_name
OPT_neuro_falls = OPT_neuro_falls.pivot(index='record_id', columns='redcap_event_name', values=[
    'number_falls', 'fall_injury'
]).reset_index()

# flatten the multi-level columns
OPT_neuro_falls.columns = ['_'.join(col).strip() for col in OPT_neuro_falls.columns.values]



## add opt parent and redo this logic
OPT_parent_falls = pd.read_excel("/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/raw/MADRS_PHQ-9_Fall_2.25.25.xlsx", engine='openpyxl', sheet_name='6473_All_Fall')

OPT_parent_falls['record_id'] = OPT_parent_falls['record_id'].str.upper()

#get rid of row 0 (descriptive row)
OPT_parent_falls = OPT_parent_falls.iloc[1:]


# filter out rows where redcap_event_name does not begin with week and ends with arm_8
OPT_parent_falls = OPT_parent_falls[
    OPT_parent_falls['redcap_event_name'].str.startswith('step_1') 
]

# Map event labels for clarity
OPT_parent_falls['redcap_event_name'] = OPT_parent_falls['redcap_event_name'].map({
    'step_1_week_2_arm_2': 'week2',
    'step_1_week_4_arm_2': 'week4',
    'step_1_week_6_arm_2': 'week6',
    'step_1_week_8_arm_2': 'week8',
    'step_1_week_10_end_arm_2': 'week10'
})

#drop rows where redcap_event_name is null
OPT_parent_falls = OPT_parent_falls.dropna(subset=['redcap_event_name'])



OPT_parent_falls = OPT_parent_falls.drop(columns=['decision_date'])


# map decision_fall_1 dictionary to df
OPT_parent_falls['decision_fall_1'] = (
    OPT_parent_falls['decision_fall_1']
    .map(fall_code)
)


fall_column_rename = {
    'decision_fall_1': 'number_falls',
    'fall_inj': 'fall_injury'}

OPT_parent_falls.rename(columns=fall_column_rename, inplace=True)

#pivot the dataframe to wide format by record_id and redcap_event_name
OPT_parent_falls = OPT_parent_falls.pivot(index='record_id', columns='redcap_event_name', values=[
    'number_falls', 'fall_injury'
]).reset_index()

# flatten the multi-level columns
OPT_parent_falls.columns = ['_'.join(col).strip() for col in OPT_parent_falls.columns.values]


OPT_falls = pd.concat([OPT_parent_falls, OPT_neuro_falls], ignore_index=True)

# rename record_id_ to record_id
OPT_falls = OPT_falls.rename(columns={'record_id_': 'record_id'})

# write to csv
OPT_falls.to_csv("/external/rprshnas01/netdata_kcni/dflab/data/BAARD/temp/processed/OPT_falls.csv", index=False)



### termination forms

## MINI


In [10]:
# load up athf for OPT_Parent

OPT_parent_mini = pd.read_csv('/projects/aabdulrasul/BAARD/BAARD/temp/raw/CoreVariable_4.17.25.csv',
    skiprows=[0],
    usecols=['id', 'mini_5', 'mini_6', 'mini_addtl_q1', 'mini_addtl_q2']
)

# rename to record_id and make all record_id uppercase
OPT_parent_mini = OPT_parent_mini.rename(columns={"id": "record_id"})
OPT_parent_mini['record_id'] = OPT_parent_mini['record_id'].str.upper()


# load up athf for OPT_Neuro
OPT_neuro_mini = pd.read_csv('/projects/aabdulrasul/BAARD/BAARD/temp/raw/OPT_N_mini.csv', 
    usecols=['record_id', 'mini_5', 'mini_6', 'mini_addtl_q1', 'mini_addtl_q2']
)
OPT_neuro_mini['record_id'] = OPT_neuro_mini['record_id'].str.upper()
# drop any rows from OPT_neuro_mini where record_id begins with CU
OPT_neuro_mini = OPT_neuro_mini[~OPT_neuro_mini['record_id'].str.startswith('CU')]

# concatenate the two dataframes
OPT_mini = pd.concat([OPT_parent_mini, OPT_neuro_mini], ignore_index=True)

OPT_mini.to_csv('/projects/aabdulrasul/BAARD/BAARD/temp/processed/OPT_mini.csv', index=False)


## ATHF


need

athf_f1_highest_trial_v2

athf_f1_total_trial_v2

athf_f1_total_score_v2

In [12]:
OPT_parent_ATHF = pd.read_excel('/projects/aabdulrasul/BAARD/BAARD/temp/raw/ATHF_step1_2025-02-12.xlsx', engine='openpyxl')

OPT_parent_ATHF = OPT_parent_ATHF.rename(columns={
    "Record ID": "record_id",
    "Total ATHF Score": "athf_f1_total_score_v2",
    "Strength of Highest Rated Trial": "athf_f1_highest_trial_v2",
    "Number of Adequate Treatment Trials": "athf_f1_total_trials_v2"
})

OPT_parent_ATHF['record_id'] = OPT_parent_ATHF['record_id'].str.upper()

# select relevant columns
OPT_parent_ATHF = OPT_parent_ATHF[['record_id', 'athf_f1_total_score_v2', 'athf_f1_highest_trial_v2', 'athf_f1_total_trials_v2']]

# load up OPT Neuro ATHF
OPT_neuro_ATHF = pd.read_csv('/projects/aabdulrasul/BAARD/BAARD/temp/raw/OPT_N_athf.csv'
)
OPT_neuro_ATHF['record_id'] = OPT_neuro_ATHF['record_id'].str.upper()
OPT_neuro_ATHF = OPT_neuro_ATHF[~OPT_neuro_ATHF['record_id'].str.startswith('CU')]

# select relevant columns
OPT_neuro_ATHF = OPT_neuro_ATHF[['record_id', 'athf_f1_total_score_v2', 'athf_f1_highest_trial_v2', 'athf_f1_total_trials_v2']]

# concatenate the two dataframes
OPT_ATHF = pd.concat([OPT_parent_ATHF, OPT_neuro_ATHF], ignore_index=True)

OPT_ATHF.to_csv('/projects/aabdulrasul/BAARD/BAARD/temp/processed/OPT_ATHF.csv', index=False)